<a href="https://colab.research.google.com/github/markjschmidt/PET_MIP/blob/main/MJS_Sonnet4_PET_MIP_Rotating_20260827.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PET Maximum Intensity Projection — Rotating MIP Viewer

Loads DICOM PET slices from GitHub (via Git LFS), builds a 3D volume, pre-computes rotating MIP projections, and displays an interactive viewer with angle and colormap controls.

**Series:** S18750 / S29811 — 182 slices, 338×338 px, 2 mm isotropic, units: Bq/mL

## Step 1 — Install dependencies and clone data

In [ ]:
# Install git-lfs (not present by default in Colab)
!apt-get install -y git-lfs -q

# Install Python packages
!pip install pydicom scipy ipywidgets -q

# Enable ipywidgets in Colab output
from google.colab import output
output.enable_custom_widget_manager()

## Step 2 — Clone repo and pull DICOM files from Git LFS

In [ ]:
import os

REPO_URL  = 'https://github.com/markjschmidt/PET_MIP'
CLONE_DIR = '/content/PET_MIP'
DICOM_DIR = f'{CLONE_DIR}/DICOM/S18750/S29811'

if not os.path.isdir(CLONE_DIR):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}
    !git -C {CLONE_DIR} lfs pull --include='DICOM/S18750/S29811/*'
else:
    print('Repo already cloned — skipping.')

# Verify
import glob
files = sorted(glob.glob(f'{DICOM_DIR}/I*'))
print(f'Found {len(files)} DICOM files.')

## Step 3 — Build 3D volume
Sorts slices by Z position and applies the DICOM rescale slope/intercept to convert to Bq/mL.

In [ ]:
import pydicom
import numpy as np

# Sort slices by Z position (ImagePositionPatient[2])
slice_list = []
for f in files:
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    z  = float(ds.ImagePositionPatient[2])
    slice_list.append((z, f))
slice_list.sort(key=lambda x: x[0])

# Load pixel data and apply rescale
volume = []
for z, f in slice_list:
    ds      = pydicom.dcmread(f)
    arr     = ds.pixel_array.astype(np.float32)
    slope   = float(getattr(ds, 'RescaleSlope',     1))
    intcpt  = float(getattr(ds, 'RescaleIntercept', 0))
    volume.append(arr * slope + intcpt)

vol = np.array(volume)   # shape: (Z, Y, X)
print(f'Volume shape : {vol.shape}')
print(f'Z range      : {slice_list[0][0]:.1f} → {slice_list[-1][0]:.1f} mm')
print(f'Pixel range  : {vol.min():.1f} → {vol.max():.1f} Bq/mL')

## Step 4 — Pre-compute rotating MIPs
Rotates the volume around the **vertical (Z) axis** at every 5°, producing 72 projections. This takes ~2–4 minutes once; the slider is instant after.

In [ ]:
from scipy.ndimage import rotate
import time

ANGLE_STEP = 5                              # degrees per step
angles     = np.arange(0, 360, ANGLE_STEP) # 72 projections
mip_stack  = np.zeros((len(angles), vol.shape[0], vol.shape[1]),
                       dtype=np.float32)    # (N_angles, Z, Y)

t0 = time.time()
for i, angle in enumerate(angles):
    # Rotate each axial slice in the XY plane, then project along the new X axis
    rot    = rotate(vol, angle, axes=(1, 2), reshape=False, order=1, prefilter=False)
    mip_stack[i] = np.max(rot, axis=2)      # (Z, Y)
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/{len(angles)} angles  ({elapsed:.0f}s elapsed)')

print(f'Done. {len(angles)} projections in {time.time()-t0:.0f}s.')
VMAX_DEFAULT = float(np.percentile(vol[vol > 0], 99))
print(f'Suggested color max (99th percentile): {VMAX_DEFAULT:.0f} Bq/mL')

## Step 5 — Interactive rotating MIP viewer
- **Angle** — rotates the projection view around the vertical axis (0°–355°)
- **Color max** — sets the upper end of the hot colormap (Bq/mL); lower = higher contrast on faint structures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display

# ── Sliders ──────────────────────────────────────────────────────────────
angle_slider = widgets.IntSlider(
    value=0, min=0, max=355, step=ANGLE_STEP,
    description='Angle (°):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=True
)

vmax_slider = widgets.FloatSlider(
    value=VMAX_DEFAULT,
    min=500.0,
    max=float(vol.max()),
    step=500.0,
    description='Color max:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.0f',
    continuous_update=False
)

vmax_label = widgets.Label(value='Bq/mL')
controls   = widgets.VBox([
    widgets.HBox([angle_slider]),
    widgets.HBox([vmax_slider, vmax_label]),
])

out = widgets.Output()

# ── Render function ───────────────────────────────────────────────────────
def render(angle, vmax):
    idx = int(round(angle / ANGLE_STEP)) % len(angles)
    img = np.flipud(mip_stack[idx])          # Z axis: superior at top

    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 10), facecolor='black')
        ax.set_facecolor('black')
        im = ax.imshow(img, cmap='hot', vmin=0, vmax=vmax, aspect='auto')

        ax.set_title(f'PET MIP  |  {angle}°  |  color max {vmax:.0f} Bq/mL',
                     color='white', fontsize=12, pad=8)
        ax.set_xlabel('← Left   Right →', color='#888', fontsize=9)
        ax.set_ylabel('Inferior ↑ Superior', color='#888', fontsize=9)
        ax.tick_params(colors='#555')
        for sp in ax.spines.values():
            sp.set_edgecolor('#333')

        # Colorbar
        cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label('Bq/mL', color='white', fontsize=9)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)

        plt.tight_layout()
        plt.show()

# ── Wire up ───────────────────────────────────────────────────────────────
interactive = widgets.interactive_output(
    render, {'angle': angle_slider, 'vmax': vmax_slider}
)

display(controls, out)
render(0, VMAX_DEFAULT)   # draw immediately

## (Optional) Export rotating MIP as animated GIF
Saves one full 360° rotation at the current color-max setting.

In [ ]:
from PIL import Image as PILImage
import io

VMAX_GIF = vmax_slider.value   # use current slider value
frames   = []

for i in range(len(angles)):
    img = np.flipud(mip_stack[i])
    fig, ax = plt.subplots(figsize=(4, 8), facecolor='black')
    ax.set_facecolor('black')
    ax.imshow(img, cmap='hot', vmin=0, vmax=VMAX_GIF, aspect='auto')
    ax.set_title(f'PET MIP  {angles[i]:.0f}°', color='white', fontsize=10)
    ax.axis('off')
    plt.tight_layout(pad=0.3)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='black')
    plt.close(fig)
    buf.seek(0)
    frames.append(PILImage.open(buf).convert('P', palette=PILImage.ADAPTIVE))

gif_path = '/content/PET_rotating_MIP.gif'
frames[0].save(
    gif_path, save_all=True, append_images=frames[1:],
    loop=0, duration=80   # 80 ms per frame ≈ 12 fps
)
print(f'Saved: {gif_path}')

# Download
from google.colab import files
files.download(gif_path)